# MNIST High Accuracy Challenge

Objetivo: Alcanzar >99.4% de accuracy en MNIST usando solo redes fully-connected (MLP).

Técnicas utilizadas:
- Batch Normalization
- Learning Rate Scheduling
- Data Augmentation
- Dropout
- Arquitectura optimizada

## Imports

In [1]:
import torch
import torchvision
import torch.nn as nn
from tqdm import tqdm
import multiprocessing
import torch.optim as optim
import torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

print("Torch version:", torch.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Torch version: 2.9.0+cu126
Device: cuda


## Data Augmentation Configuration

In [2]:
train_transform = transforms.Compose([
    # Data Augmentation más agresivo: Rotación, Traslación, Escala y Cizallamiento (Shear)
    # Esto es clave para que el modelo generalice ante variaciones de escritura
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1), shear=10),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)) # Normalización estándar de MNIST
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

## Dataset Class

In [3]:
class MNIST_dataset(Dataset):
    
    def __init__(self, partition="train", transform=None):
        print("\nLoading MNIST ", partition, " Dataset...")
        self.partition = partition
        self.transform = transform
        
        if self.partition == "train":
            self.data = torchvision.datasets.MNIST('.data/', train=True, download=True)
        else:
            self.data = torchvision.datasets.MNIST('.data/', train=False, download=True)
        
        print("\tTotal Len.: ", len(self.data), "\n", 50*"-")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image = self.data[idx][0]
        image = self.transform(image)
        image = image.view(-1)

        label = self.data[idx][1]
        # Devolvemos el índice de la clase (Long) en lugar de One-Hot
        # Esto es necesario para usar label_smoothing en CrossEntropyLoss de forma eficiente
        label = torch.tensor(label, dtype=torch.long)

        return {"idx": idx, "img": image, "label": label}

## Neural Network with Batch Normalization and Dropout

In [4]:
class Net(nn.Module):
    def __init__(self, sizes=[[784, 1024], [1024, 1024], [1024, 1024], [1024, 512], [512, 10]], 
                 dropout_rate=0.3, criterion=None):
        super(Net, self).__init__()
        
        self.layers = nn.ModuleList()
        
        for i in range(len(sizes) - 1):
            dims = sizes[i]
            self.layers.append(nn.Linear(dims[0], dims[1]))
            self.layers.append(nn.BatchNorm1d(dims[1]))
            self.layers.append(nn.ReLU())
            self.layers.append(nn.Dropout(dropout_rate))
        
        dims = sizes[-1]
        self.classifier = nn.Linear(dims[0], dims[1])
        self.criterion = criterion
        
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                # He initialization (Kaiming) es mejor para ReLU
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x, y=None):
        for layer in self.layers:
            x = layer(x)
        x = self.classifier(x)
        
        if y is not None:
            loss = self.criterion(x, y)
            return loss, x
        return x

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## Load Data and Create DataLoaders

In [5]:
train_dataset = MNIST_dataset(partition="train", transform=train_transform)
test_dataset = MNIST_dataset(partition="test", transform=test_transform)

batch_size = 100
num_workers = multiprocessing.cpu_count() - 1
print("Num workers", num_workers)

train_dataloader = DataLoader(train_dataset, batch_size, shuffle=True, num_workers=num_workers)
test_dataloader = DataLoader(test_dataset, batch_size, shuffle=False, num_workers=num_workers)


Loading MNIST  train  Dataset...
	Total Len.:  60000 
 --------------------------------------------------

Loading MNIST  test  Dataset...
	Total Len.:  10000 
 --------------------------------------------------
Num workers 1


## Initialize Model and Training Configuration

In [6]:
# Usamos Label Smoothing: ayuda a que el modelo no sea "demasiado confiado" y generalice mejor
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

num_classes = 10
# Arquitectura mas ancha para mayor capacidad (Over-parameterization)
net = Net(
    sizes=[
        [784, 1500], 
        [1500, 1500], 
        [1500, 1000], 
        [1000, 500], 
        [500, num_classes]
    ], 
    dropout_rate=0.2, # Subimos dropout ligeramente al aumentar el tamaño
    criterion=criterion
)

print(net)
print("Params: ", count_parameters(net))

# Vuelta a SGD con Momentum (suele generalizar mejor en el limite que Adam para vision)
# Pero con un LR inicial mas alto y un schedule agresivo
# Bajamos un poco el weight_decay (de 5e-4 a 1e-4) porque ya tenemos mucho Data Augmentation y Dropout
optimizer = optim.SGD(net.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4)
# scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[30, 50, 60], gamma=0.1)
# Cambiamos a ReduceLROnPlateau para bajar el LR automáticamente cuando la accuracy se estanque
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.2, patience=3, min_lr=1e-6)

net = net.to(device)
epochs = 60 # Reducimos épocas ya que con mejor data augmentation deberíamos converger mejor

Net(
  (layers): ModuleList(
    (0): Linear(in_features=784, out_features=1500, bias=True)
    (1): BatchNorm1d(1500, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=1500, out_features=1500, bias=True)
    (5): BatchNorm1d(1500, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=1500, out_features=1000, bias=True)
    (9): BatchNorm1d(1000, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.2, inplace=False)
    (12): Linear(in_features=1000, out_features=500, bias=True)
    (13): BatchNorm1d(500, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (14): ReLU()
    (15): Dropout(p=0.2, inplace=False)
  )
  (classifier): Linear(in_features=500, out_features=10, bias=True)
  (criterion): CrossEntropyLoss()
)
Params:  5444510


## Training Loop

In [7]:
print("\n---- Start Training ----")
best_accuracy = -1
best_epoch = 0

for epoch in range(epochs):
    
    # TRAIN NETWORK
    train_loss, train_correct = 0, 0
    net.train()
    
    for batch in train_dataloader:
        images = batch["img"].to(device)
        labels = batch["label"].to(device)
        ids = batch["idx"].to('cpu').numpy()
        
        optimizer.zero_grad()
        loss, outputs = net(images, labels)
        loss.backward()
        optimizer.step()
        
        # labels ya son índices, no hace falta argmax
        # labels = torch.argmax(labels, dim=1) 
        pred = torch.argmax(outputs, dim=1)
        train_correct += pred.eq(labels).sum().item()
        train_loss += loss.item()

    # scheduler.step() # Movido al final para ReduceLROnPlateau
    # Corregimos la normalizacion del loss (promedio por batch en lugar de por sample total)
    train_loss /= len(train_dataloader) 
    train_accuracy = 100. * train_correct / len(train_dataloader.dataset)
    
    # TEST NETWORK
    test_loss, test_correct = 0, 0
    net.eval()
    
    with torch.no_grad():
        for batch in test_dataloader:
            images = batch["img"].to(device)
            labels = batch["label"].to(device)
            ids = batch["idx"].to('cpu').numpy()
            
            outputs = net(images)
            test_loss += criterion(outputs, labels).item()
            
            # labels ya son índices
            # labels = torch.argmax(labels, dim=1)
            pred = torch.argmax(outputs, dim=1)
            test_correct += pred.eq(labels).sum().item()
    
    test_loss /= len(test_dataloader)
    test_accuracy = 100. * test_correct / len(test_dataloader.dataset)
    
    # Actualizamos el scheduler basándonos en la accuracy de test
    scheduler.step(test_accuracy)
    
    print("[Epoch {:2d}] Train: {:.2f}% | Test: {:.2f}% | Loss: {:.4f} | LR: {:.5f}".format(
        epoch + 1, train_accuracy, test_accuracy, test_loss, optimizer.param_groups[0]['lr']
    ))
    
    if test_accuracy > best_accuracy:
        best_accuracy = test_accuracy
        best_epoch = epoch
        torch.save(net.state_dict(), "best_model_high_acc.pt")

print("\nBEST TEST ACCURACY: ", best_accuracy, " in epoch ", best_epoch)


---- Start Training ----
[Epoch  1] Train: 76.45% | Test: 95.27% | Loss: 0.6675 | LR: 0.10000
[Epoch  1] Train: 76.45% | Test: 95.27% | Loss: 0.6675 | LR: 0.10000
[Epoch  2] Train: 87.85% | Test: 96.46% | Loss: 0.6205 | LR: 0.10000
[Epoch  2] Train: 87.85% | Test: 96.46% | Loss: 0.6205 | LR: 0.10000
[Epoch  3] Train: 90.66% | Test: 97.04% | Loss: 0.5969 | LR: 0.10000
[Epoch  3] Train: 90.66% | Test: 97.04% | Loss: 0.5969 | LR: 0.10000
[Epoch  4] Train: 91.89% | Test: 97.69% | Loss: 0.5834 | LR: 0.10000
[Epoch  4] Train: 91.89% | Test: 97.69% | Loss: 0.5834 | LR: 0.10000
[Epoch  5] Train: 93.07% | Test: 98.06% | Loss: 0.5704 | LR: 0.10000
[Epoch  5] Train: 93.07% | Test: 98.06% | Loss: 0.5704 | LR: 0.10000
[Epoch  6] Train: 93.72% | Test: 98.10% | Loss: 0.5628 | LR: 0.10000
[Epoch  6] Train: 93.72% | Test: 98.10% | Loss: 0.5628 | LR: 0.10000
[Epoch  7] Train: 94.22% | Test: 98.24% | Loss: 0.5582 | LR: 0.10000
[Epoch  7] Train: 94.22% | Test: 98.24% | Loss: 0.5582 | LR: 0.10000
[Epoch  

## Load Best Model and Final Evaluation

In [9]:
net.load_state_dict(torch.load("best_model_high_acc.pt"))

test_loss, test_correct = 0, 0
net.eval()

with torch.no_grad():
    with tqdm(iter(test_dataloader), desc="Test " + str(epoch), unit="batch") as tepoch:
        for batch in tepoch:
            images = batch["img"].to(device)
            labels = batch["label"].to(device)
            ids = batch["idx"].to('cpu').numpy()
            
            outputs = net(images)
            test_loss += criterion(outputs, labels).item()
            
            # labels ya son índices
            # labels = torch.argmax(labels, dim=1)
            pred = torch.argmax(outputs, dim=1)
            test_correct += pred.eq(labels).sum().item()

test_loss /= len(test_dataloader)
test_accuracy = 100. * test_correct / len(test_dataloader.dataset)
print(f"Final best acc: {test_accuracy:.1f}")

Test 59: 100%|██████████| 100/100 [00:03<00:00, 30.04batch/s]

Final best acc: 99.4
